# Multimodal Evaluation - Gemini 2.5 Flash (PANNs)

Queries condition C SQL on combined DB. Writes xlsx to data/analysis_gemini_2_5_flash_panns/.


In [ ]:
import sys, os, sqlite3, json, subprocess
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)
ABLATION_DIR = ROOT / 'data' / 'ablation_gemini_2_5_flash_vlm_rag_reid_panns'
ANALYSIS_DIR = ROOT / 'data' / 'analysis_gemini_2_5_flash_vlm_rag_reid_panns'
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'
from service.impl.events_service_impl import queries_for_condition
from utils.database import setup_database, ensure_default_db
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')


In [ ]:
# ── Audio configs ──
AUDIO_CONFIGS = [
    ('w1_0s_h1_0s', 1.0, 1.00, 'ab0'),
    ('w1_0s_h0_5s', 1.0, 0.50, 'ab1'),
    ('w2_5s_h2_5s', 2.5, 2.50, 'ab2'),
    ('w2_5s_h1_25s', 2.5, 1.25, 'ab3'),
    ('w5_0s_h5_0s', 5.0, 5.00, 'ab4'),
    ('w5_0s_h2_5s', 5.0, 2.50, 'ab5'),
    ('w10_0s_h10_0s', 10.0, 10.00, 'ab6'),
    ('w10_0s_h5_0s', 10.0, 5.00, 'ab7'),
]

# ── Source DBs ──
visual_db = ROOT / 'data' / 'ablation_gemini_2_5_flash_vlm_rag_reid' / 'gemini_2_5_flash_vlm_rag_reid.db'
if not visual_db.exists():
    raise FileNotFoundError(f'Visual DB missing: {visual_db}')
missing = []
for win_label, window, hop, prefix in AUDIO_CONFIGS:
    audio_db = ROOT / 'data' / 'ablation_panns' / f'panns_{win_label}.db'
    if not audio_db.exists():
        missing.append(str(audio_db))
if missing:
    raise FileNotFoundError(f'Audio DBs missing: {missing}')
print(f'Visual DB: {visual_db}')
print(f'{len(AUDIO_CONFIGS)} audio configs: {[c[0] for c in AUDIO_CONFIGS]}')


In [ ]:
# ── Create combined DB for each audio config ──
ablation_dir = ROOT / 'data' / 'ablation_gemini_2_5_flash_vlm_rag_reid_panns'
ablation_dir.mkdir(parents=True, exist_ok=True)
for win_label, window, hop, prefix in AUDIO_CONFIGS:
    audio_db = ROOT / 'data' / 'ablation_panns' / f'panns_{win_label}.db'
    combined_db = ablation_dir / f'gemini_2_5_flash_vlm_rag_reid_panns_{win_label}.db'
    conn, cur = setup_database(combined_db)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA foreign_keys=OFF')
    for _tbl in ('AudioPerInterval', 'VisualPerInterval', 'VisualParticipant',
                  'VisualPerFrame', 'VisualRelation'):
        conn.execute(f'DELETE FROM {_tbl}')
    audio_conn = sqlite3.connect(str(audio_db))
    adf = pd.read_sql_query('SELECT * FROM AudioPerInterval', audio_conn)
    audio_conn.close()
    adf['AnalysisID'] = adf['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if 'ab' in x else x)
    adf.to_sql('AudioPerInterval', conn, if_exists='append', index=False)
    visual_conn = sqlite3.connect(str(visual_db))
    vdf = pd.read_sql_query('SELECT * FROM VisualPerInterval', visual_conn)
    participant_df = pd.read_sql_query('SELECT * FROM VisualParticipant', visual_conn)
    perframe_df = pd.read_sql_query('SELECT * FROM VisualPerFrame', visual_conn)
    rel_df = pd.read_sql_query('SELECT * FROM VisualRelation', visual_conn)
    visual_conn.close()
    vdf['AnalysisID'] = vdf['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    perframe_df['AnalysisID'] = perframe_df['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    rel_df['AnalysisID'] = rel_df['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    vdf.to_sql('VisualPerInterval', conn, if_exists='append', index=False)
    participant_df.to_sql('VisualParticipant', conn, if_exists='append', index=False)
    perframe_df.to_sql('VisualPerFrame', conn, if_exists='append', index=False)
    rel_df.to_sql('VisualRelation', conn, if_exists='append', index=False)
    conn.commit()
    size_kb = os.path.getsize(combined_db) // 1024
    print(f'Combined DB created: {combined_db} ({size_kb} KiB)')
    conn.close()
print('All combined DBs ready.')


In [ ]:
# ── Load GT ──
gt = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_visual = gt[gt['modality'] == 'visual'].dropna(subset=['scene'])
gt_visual['scene'] = gt_visual['scene'].astype(int)
gt_audio = gt[gt['modality'] == 'audio'].dropna(subset=['scene'])
gt_audio['scene'] = gt_audio['scene'].astype(int)
gt_audio['class'] = gt_audio['class'].apply(lambda c: c.strip().lower().replace(' ', '_') if isinstance(c, str) else c)
expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')
print(f'GT: visual={len(gt_visual)} audio={len(gt_audio)} expected={len(expected_df)}')


In [ ]:
# ── DELTAS for condition C ──
# Defaults are sourced from each event's authored ISEQL in the event
# registry (single source of truth), like the visual eval notebooks, NOT
# hardcoded, so they stay in sync with the DB event specs (e.g.
# suspicious_near_vehicle replaces potential_car_theft).

# Detect video fps per video (like the frontend); fallback 24.
def detect_fps(video_path: str, default: int = 24) -> int:
    try:
        probe = subprocess.check_output(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=avg_frame_rate,r_frame_rate",
             "-of", "json", video_path], timeout=10, stderr=subprocess.DEVNULL)
        info = json.loads(probe)
        for key in ("avg_frame_rate", "r_frame_rate"):
            fps_str = info["streams"][0].get(key, "")
            if fps_str and "/" in fps_str:
                num, den = fps_str.split("/")
                fps = int(num) // int(den) if int(den) else 0
                if fps > 0:
                    return fps
    except Exception:
        pass
    return default

# Delta / rho in seconds, zeta as strictness operators. Converted to frames
# per video using its detected fps (like the frontend).
from service.impl.events_service_impl import default_deltas_for, derive_delta_fields
from service.impl.event_registry_service_impl import EventRegistryServiceImpl as _Reg

ensure_default_db()
_evt_conn = sqlite3.connect(str(ROOT / 'data/analysis.db'))
_evt_conn.row_factory = sqlite3.Row
_reg = _Reg()
_DELTA_FIELDS = ('delta_visual','delta_audio','epsilon_visual','epsilon_audio',
                 'eta_visual','eta_audio','zeta_visual','zeta_audio','rho_visual','rho_audio')
DEFAULT_DELTAS = {}
for _cond in ('A','B','C'):
    for _e in _reg.list_events(_evt_conn, condition=_cond):
        _f = derive_delta_fields(_e.model_json)
        DEFAULT_DELTAS.update(default_deltas_for(_e.model_json, _e.id, _f))
_evt_conn.close()

def params_for_scene(scene) -> tuple[dict, int]:
    fps = detect_fps(str(VIDEO_DIR / f'scene{scene}.mp4'))
    def frames(d: dict) -> dict:
        return {
            k: (round(v * fps) if isinstance(v, (int, float)) and not isinstance(v, bool) else v)
            for k, v in d.items()
        }
    return frames(DEFAULT_DELTAS), fps


In [ ]:
# ── Multimodal event evaluation (all audio configs) ──
audio_evts = set(expected_df[expected_df['audio'].notna() & (expected_df['audio'] != '')]['event'].unique())
all_scenes = sorted(expected_df['scene'].unique())
summary_rows = []

for win_label, window, hop, prefix in AUDIO_CONFIGS:
    combined_db = ablation_dir / f'gemini_2_5_flash_vlm_rag_reid_panns_{win_label}.db'
    conn = sqlite3.connect(str(combined_db))
    event_rows = []
    for _, row in expected_df.iterrows():
        aid = f'gemini_2_5_flash_panns_s{row["scene"]}'
        evt = row['event']
        try:
            params, fps = params_for_scene(row["scene"])
            sql_map = queries_for_condition('C', params, analysis_id=aid, fps=fps)
            sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
            df = pd.read_sql_query(sql, conn)
            det = not df.empty
            result = 'TP' if det else 'FN'
        except Exception as e:
            print(f'  {evt} query failed: {e}')
            det = False; result = 'ERROR'
        parts = []
        if evt in audio_evts:
            all_audio = conn.execute('SELECT AudioClass, StartFrame, EndFrame FROM AudioPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
            if all_audio:
                parts += [f'audio: {s}({sf}-{ef})' for s, sf, ef in all_audio]
        all_visual = conn.execute('SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
        if all_visual:
            parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
        if det and not parts:
            parts = [f'{evt} (query matched)']
        event_rows.append({'scene': row['scene'], 'event': evt, 'detected': 'YES' if det else 'NO', 'result': result, 'relations': ', '.join(parts)})

    # FP pass
    for evt in sorted(expected_df['event'].unique()):
        pos_scenes = set(expected_df[expected_df['event'] == evt]['scene'])
        for scene in all_scenes:
            if scene in pos_scenes:
                continue
            aid = f'gemini_2_5_flash_panns_s{scene}'
            try:
                params, fps = params_for_scene(scene)
                sql_map = queries_for_condition('C', params, analysis_id=aid, fps=fps)
                sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
                df = pd.read_sql_query(sql, conn)
                if not df.empty:
                    parts = []
                    if evt in audio_evts:
                        all_audio = conn.execute('SELECT AudioClass, StartFrame, EndFrame FROM AudioPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                        if all_audio:
                            parts += [f'audio: {s}({sf}-{ef})' for s, sf, ef in all_audio]
                    all_visual = conn.execute('SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                    if all_visual:
                        parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
                    event_rows.append({'scene': scene, 'event': evt, 'detected': 'YES', 'result': 'FP', 'relations': ', '.join(parts) if parts else '|'})
            except:
                pass
    vpi = conn.execute('SELECT COUNT(*) FROM VisualPerInterval').fetchone()[0]
    api = conn.execute('SELECT COUNT(*) FROM AudioPerInterval').fetchone()[0]
    conn.close()

    edf = pd.DataFrame(event_rows)
    edf['relations'] = edf['relations'].fillna('')
    metrics = []
    for evt in sorted(expected_df['event'].unique()):
        sub = edf[edf['event'] == evt]
        tpp = len(sub[sub['result'] == 'TP']); fpp = len(sub[sub['result'] == 'FP']); fnn = len(sub[sub['result'] == 'FN'])
        support = tpp + fnn
        p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
        r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
        f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
        metrics.append({'event': evt, 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f1,3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support})
    metrics_df = pd.DataFrame(metrics)
    xlsx_path = ANALYSIS_DIR / f'multimodal_event_eval_gemini_2_5_flash_vlm_rag_reid_panns_{win_label}.xlsx'
    with pd.ExcelWriter(xlsx_path) as writer:
        metrics_df.to_excel(writer, sheet_name='Summary', index=False)
        for sc in all_scenes:
            sc_df = edf[edf['scene'] == sc]
            if not sc_df.empty:
                sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)
    print(f'Written: {xlsx_path}')

    tp = len(edf[edf['result'] == 'TP']); fp = len(edf[edf['result'] == 'FP']); fn = len(edf[edf['result'] == 'FN'])
    support = tp + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    summary_rows.append({'visual': 'gemini_2_5_flash', 'reid': True, 'audio': 'panns', 'window': window, 'hop': hop,
        'precision': round(precision,3), 'recall': round(recall,3), 'f1': round(f1,3),
        'TP': tp, 'FP': fp, 'FN': fn, 'support': support, 'VPI': vpi, 'API': api})
    print(f'  [{win_label}] Summary: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn}')

summary_df = pd.DataFrame(summary_rows)
summary_df.to_excel(ANALYSIS_DIR / 'summary.xlsx', index=False)
print(f'\n=== Pair summary (all configs) ===')
print(summary_df.to_string(index=False))
